# Speaking AI Demo
## Notebook 2: TensorFlow + NumPy Speaking AI

Notebook ini mendemonstrasikan model Text-to-Speech (TTS)
menggunakan TensorFlow dan NumPy untuk audio processing.

In [ ]:
import sys
import os
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

print("TensorFlow version:", __import__('tensorflow').__version__)
print("NumPy version:", np.__version__)
print("Ready!")

## 1. Audio Preprocessing

In [ ]:
from src.speaking_ai import AudioPreprocessor

preprocessor = AudioPreprocessor(sample_rate=22050, n_mfcc=13)

# Generate sine wave
sine = preprocessor.generate_sine_wave(frequency=440, duration=0.5)
print(f"Sine wave: {len(sine)} samples")

# Generate complex tone (C major chord)
chord = preprocessor.generate_complex_tone(
    frequencies=[261.63, 329.63, 392.00],  # C4, E4, G4
    duration=1.0
)
print(f"C major chord: {len(chord)} samples")

# Add noise
noisy = preprocessor.add_noise(chord, noise_level=0.02)

# Voice activity detection
vad = preprocessor.detect_voice_activity(noisy, threshold=0.01)
print(f"VAD result: {len(vad)} frames, {np.sum(vad)} voice frames")

## 2. MFCC Feature Extraction

In [ ]:
mfcc = preprocessor.compute_mfcc_numpy(sine)
print(f"MFCC shape: {mfcc.shape}")

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.imshow(mfcc.T, aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='MFCC Value')
plt.xlabel('Frame')
plt.ylabel('MFCC Coefficient')
plt.title('MFCC Features')
plt.tight_layout()
plt.show()

## 3. Speaking AI System

In [ ]:
from src.speaking_ai import SpeakingAI

ai = SpeakingAI(
    sample_rate=22050,
    vocab_size=100,
    max_text_length=30,
    audio_feature_dim=80
)

# Sample texts
texts = [
    "hello world",
    "machine learning",
    "deep learning models",
    "artificial intelligence",
    "natural language",
    "neural networks",
    "text to speech",
    "voice synthesis",
    "audio processing",
    "data science",
]

# Generate training data
X_text, X_audio = ai.generate_training_data(texts)
print(f"Training data: {X_text.shape}, {X_audio.shape}")

## 4. Train TTS Model

In [ ]:
# Build and train model
ai.tts_model.build_lightweight_model()

history = ai.tts_model.train(
    X_text, X_audio,
    epochs=20,
    batch_size=4,
    validation_split=0.2
)

# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history['loss'], label='Training Loss')
if history['val_loss']:
    plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('TTS Model Training')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

## 5. Generate Speech

In [ ]:
# Generate speech from text
test_text = "hello machine learning"
mel_output = ai.speak(test_text)

# Visualize generated mel-spectrogram
plt.figure(figsize=(12, 4))
plt.imshow(mel_output.T, aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='Amplitude')
plt.xlabel('Frame')
plt.ylabel('Mel Bin')
plt.title(f'Generated Speech: "{test_text}"')
plt.tight_layout()
plt.show()

print(f"Generated {mel_output.shape[0]} frames of audio features")